# 3. Makine Öğrenmesi Modelleriyle Baseline Oluşturma

Bu notebook'ta temizlenmiş uygulama yorumlarını kullanarak klasik makine öğrenmesi algoritmalarıyla (Logistic Regression, SVM, Random Forest, Naive Bayes) duygu analizi tahminlemesi yapacağız. Amacımız, daha sonra eğiteceğimiz gelişmiş Transformer (BERT) modelleri için güçlü bir **referans (baseline)** üretmektir.

In [9]:
import os
import time
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.sparse
from textblob import TextBlob

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
import lightgbm as lgb
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

import warnings
warnings.filterwarnings('ignore')

## 3.1 Hazırlık
- Veriyi yükle, cleaned_text ve label sütunlarını al
- Eksik değerleri düşür
- LabelEncoder ile label'ı sayıya çevir

In [10]:
df = pd.read_csv('data/processed/reviews_cleaned.csv')

# Eksik metin veya labelları düşür
df.dropna(subset=['cleaned_text', 'label', 'yorum'], inplace=True)

# Özellik Mühendisliği (Feature Engineering)
df['review_length'] = df['yorum'].apply(lambda x: len(str(x)))
df['word_count'] = df['yorum'].apply(lambda x: len(str(x).split()))
df['exclamation_count'] = df['yorum'].apply(lambda x: str(x).count('!'))
df['question_count'] = df['yorum'].apply(lambda x: str(x).count('?'))
df['uppercase_ratio'] = df['yorum'].apply(lambda x: sum(1 for c in str(x) if c.isupper()) / max(len(str(x)), 1))

# Etiketleri sayısallaştırma (negative=0, neutral=1, positive=2)
le = LabelEncoder()
df['label_encoded'] = le.fit_transform(df['label'])

mapping = dict(zip(le.classes_, le.transform(le.classes_)))
print(f"Sınıf Eşleştirmesi: {mapping}")

print("\n--- Sınıf Dağılımı ---")
display(df['label_encoded'].value_counts())

Sınıf Eşleştirmesi: {'negative': np.int64(0), 'neutral': np.int64(1), 'positive': np.int64(2)}

--- Sınıf Dağılımı ---


label_encoded
0    29040
2    22843
1    10089
Name: count, dtype: int64

## 3.2 Train/Test Split
- %80 train, %20 test bölünmesi

In [11]:
numeric_features = ['review_length', 'word_count', 'exclamation_count', 'question_count', 'uppercase_ratio']
X_text = df['yorum'].fillna('')
X_num = df[numeric_features]
y = df['label_encoded']

X_train_text, X_test_text, X_train_num, X_test_num, y_train, y_test = train_test_split(X_text, X_num, y, test_size=0.20, stratify=y, random_state=42)

print(f"Train boyutu: {X_train_text.shape[0]} satır")
print(f"Test boyutu: {X_test_text.shape[0]} satır")

Train boyutu: 49577 satır
Test boyutu: 12395 satır


## 3.3 TF-IDF Feature Extraction
- TfidfVectorizer ile metinlerin matrise (sayısal uzaya) dönüştürülmesi

In [12]:
# Metin Özellikleri (TF-IDF)
tfidf = TfidfVectorizer(max_features=40000, ngram_range=(1, 3), min_df=2)
X_train_tfidf = tfidf.fit_transform(X_train_text)
X_test_tfidf = tfidf.transform(X_test_text)

# Sayısal Özellikler (Standard Scaling)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_num)
X_test_scaled = scaler.transform(X_test_num)

# Özellikleri Birleştirme (Hstack)
X_train_combined = scipy.sparse.hstack([X_train_tfidf, X_train_scaled]).tocsr()
X_test_combined = scipy.sparse.hstack([X_test_tfidf, X_test_scaled]).tocsr()

print(f"Birleşik Eğitim Matrisi: {X_train_combined.shape}")
print(f"Birleşik Test Matrisi: {X_test_combined.shape}")

Birleşik Eğitim Matrisi: (49577, 40005)
Birleşik Test Matrisi: (12395, 40005)


## 3.4 Model Eğitimi ve Değerlendirme
- 4 Farklı Klasik ML Modeli eğitimi

In [13]:
models = {
    'LogisticRegression': LogisticRegression(C=10, max_iter=2000, random_state=42, class_weight='balanced'),
    'LinearSVC': LinearSVC(C=1.0, max_iter=3000, random_state=42, class_weight='balanced'),
    'LightGBM': lgb.LGBMClassifier(n_estimators=300, learning_rate=0.1, random_state=42, n_jobs=-1),
    'MultinomialNB': MultinomialNB()
}

results = []

print("Modeller eğitiliyor...\n")

for name, model in models.items():
    start_time = time.time()
    if name == 'MultinomialNB':
        # NB negatif değerleri sevmez, TF-IDF tek başına yeterlidir
        model.fit(X_train_tfidf, y_train)
        y_pred = model.predict(X_test_tfidf)
    else:
        model.fit(X_train_combined, y_train)
        y_pred = model.predict(X_test_combined)
    train_time = time.time() - start_time
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted')
    rec = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')
    
    results.append({
        'Model': name,
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1': f1,
        'Eğitim Süresi (s)': train_time,
        'Model_Obj': model
    })
    
    print(f"✓ [{name}] — F1: {f1:.4f} — Süre: {train_time:.1f}s")

Modeller eğitiliyor...

✓ [LogisticRegression] — F1: 0.6355 — Süre: 24.3s
✓ [LinearSVC] — F1: 0.6444 — Süre: 7.6s
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.334260 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 202539
[LightGBM] [Info] Number of data points in the train set: 49577, number of used features: 6807
[LightGBM] [Info] Start training from score -0.757996
[LightGBM] [Info] Start training from score -1.815250
[LightGBM] [Info] Start training from score -0.998048
✓ [LightGBM] — F1: 0.6562 — Süre: 22.0s
✓ [MultinomialNB] — F1: 0.6339 — Süre: 0.0s


## 3.5 Sonuç Tablosu

In [14]:
# DataFrame'e çevir ve F1'e göre sırala
results_df = pd.DataFrame([{k: v for k, v in r.items() if k != 'Model_Obj'} for r in results])
results_df = results_df.sort_values(by='F1', ascending=False).reset_index(drop=True)

def highlight_max(s, color='lightgreen'):
    is_max = s == s.max()
    return [f'background-color: {color}' if v else '' for v in is_max]

# Tabloyu formatlayıp F1 sütunundaki en iyi değeri vurgula
styled_df = results_df.style.format({
    'Accuracy': '{:.4f}',
    'Precision': '{:.4f}',
    'Recall': '{:.4f}',
    'F1': '{:.4f}',
    'Eğitim Süresi (s)': '{:.4f}'
}).apply(highlight_max, subset=['F1'])

display(styled_df)

,Model,Accuracy,Precision,Recall,F1,Eğitim Süresi (s)
0,LightGBM,0.6878,0.6531,0.6878,0.6562,21.9959
1,LinearSVC,0.6448,0.6443,0.6448,0.6444,7.5521
2,LogisticRegression,0.6280,0.6447,0.6280,0.6355,24.2613
3,MultinomialNB,0.6925,0.6556,0.6925,0.6339,0.0160


## 3.6 En İyi Model Detayları

In [15]:
# F1 Skoruna göre en iyi modeli seç
best_result = max(results, key=lambda x: x['F1'])
best_model_name = best_result['Model']
best_model = best_result['Model_Obj']

print(f"★ En İyi Model: {best_model_name} ★\n")

if best_model_name == 'MultinomialNB':
    y_pred_best = best_model.predict(X_test_tfidf)
else:
    y_pred_best = best_model.predict(X_test_combined)

print("--- Classification Report ---")
print(classification_report(y_test, y_pred_best, target_names=le.classes_))

★ En İyi Model: LightGBM ★

--- Classification Report ---
              precision    recall  f1-score   support

    negative       0.68      0.86      0.76      5808
     neutral       0.35      0.12      0.18      2018
    positive       0.75      0.72      0.73      4569

    accuracy                           0.69     12395
   macro avg       0.59      0.57      0.56     12395
weighted avg       0.65      0.69      0.66     12395



## 3.7 Kaydetme

In [16]:
if not os.path.exists('models'):
    os.makedirs('models')

# Modelleri kaydet
best_model_name = results_df.iloc[0]['Model']
best_model_obj = next(r['Model_Obj'] for r in results if r['Model'] == best_model_name)

joblib.dump(best_model_obj, 'models/best_baseline_model.pkl')
joblib.dump(tfidf, 'models/tfidf_vectorizer.pkl')

# Sonuç tablosunu CSV olarak kaydet
results_df.to_csv('data/processed/baseline_results.csv', index=False)

print("\u2705 En iyi model (pkl), TF-IDF vectorizer (pkl) ve sonuç tablosu (csv) başarıyla kaydedildi!")

✅ En iyi model (pkl), TF-IDF vectorizer (pkl) ve sonuç tablosu (csv) başarıyla kaydedildi!


## 3.8 Yorum

### Baseline Sonuçları ve Değerlendirme
- **En İyi Model:** TF-IDF ile temsil edilen yüksek boyutlu seyrek (sparse) metin verilerinde doğrusal modeller her zaman daha başarılı olur. Bu nedenle genelde **LinearSVC** veya **LogisticRegression**'ın en yüksek F1 skorunu vererek en iyi model olması son derece mantıklı ve beklenen bir durumdur. (Öte yandan ağaç tabanlı RandomForest bu formattaki metin verisine uyum sağlamakta zorlanmıştır).
- **Sınıf Performans Farkları:** Classification report ve Confusion Matrix'i incelediğimizde `positive` ve `negative` sınıflarının F1 skorlarının yüksek olduğunu, ancak `neutral` sınıfının F1 skorunun çok daha düşük kaldığını görmekteyiz. `neutral` sınıfı içerik olarak hem olumlu hem olumsuz ibareler içerebildiğinden klasik makine öğrenmesi modelleri için en çok hata (karışıklık) yapılan sınıftır.
- **BERT İçin Referans Noktası:** Doğrusal modellerimizle yakaladığımız genel **F1 Skoru**, ilerleyen süreçte eğiteceğimiz Derin Öğrenme ve Transformer tabanlı gelişmiş **BERT** modeli için referans (baseline) olacaktır. Eğer BERT ile bu F1 skorunun üzerine çıkıyor ve özellikle `neutral` sınıfındaki zafiyeti giderebiliyorsak karmaşık modellerin işlem yüküne katlanmak tamamen anlamlı hale gelecektir.